# Semana 8 — Regex ligera

**Programación para Ciencia de Datos**<br>
Sesiones: martes 13, miércoles 14 y viernes 16 de octubre de 2026

Las semanas 6 y 7 trataron columnas **numéricas**. Ahí un valor sucio es un
número fuera de rango o una celda vacía, y una máscara booleana resuelve el
problema. Esta semana van las columnas de **texto** de tu dataset —`email`,
`telefono`, `timestamp`, `direccion`— donde el problema es otro: el valor
*parece* correcto pero el formato no lo es, y no hay ninguna comparación `>` que
te lo diga.

Una **expresión regular** (*regex*) es un patrón que describe la **forma** de una
cadena. Con ella puedes preguntar tres cosas, y son las tres de la semana:

| Pregunta | Función |
|---|---|
| ¿Esta cadena tiene la forma correcta? | `re.fullmatch` / `re.search` |
| ¿Qué pedazos de esta cadena cumplen el patrón? | `re.findall` / `re.finditer` |
| Cámbiame los pedazos que cumplen por otra cosa | `re.sub` |

> **De aquí sale parte de la Práctica 4**, donde vas a validar `email` y
> `telefono` de tu dataset y reportar cuántos valores de cada columna están mal
> formados. La otra mitad de la P4 es pandas (semanas 9 y 10): regex es la
> herramienta, pandas es donde la vas a aplicar a 100,000 filas de un jalón con
> el *accessor* `.str`. Los patrones que escribas esta semana se reusan tal cual.

> **Lo que esta semana *no* cubre:** *lookahead* y *lookbehind*, referencias
> hacia atrás dentro del mismo patrón, `re.split`, `re.VERBOSE` y la librería
> `regex` de PyPI. Nada de eso te hace falta para la P4. "Ligera" está en el
> nombre del tema a propósito: el 90% del trabajo real se hace con lo de este
> notebook.

**Mapa de la semana**

| Sesión | Tema | Lo que vas a poder hacer al final |
|---|---|---|
| 1 (mar 13) | Metacaracteres, clases de caracteres, cuantificadores y anclas | Escribir un patrón que describa la forma de un dato, y probarlo |
| 2 (mié 14) | Grupos de captura, `search`, `findall`, `sub`, `compile` | Extraer las partes de un valor y reescribirlas |
| 3 (vie 16) | Taller: validar correos, extraer teléfonos, normalizar fechas | Las tres validaciones que pide la P4 |

> **Cómo usar este notebook.** Los ejemplos van sobre listas chicas de correos,
> teléfonos y fechas escritas aquí a mano, no sobre tu CSV. Son chicas para que
> puedas verificar con los ojos qué acepta y qué rechaza cada patrón: un
> validador que no probaste contra casos malos no es un validador, es una
> suposición. Aplicarlo a tus 100,000 filas es tu trabajo y va en los ejercicios.


### Los datos de hoy

Tres listas, con la misma clase de basura que tiene tu dataset. Los valores están
inventados, pero los **tipos de error** son exactamente los que vas a encontrar
en tus columnas `email`, `telefono` y `timestamp`.


In [21]:
import re

correos = [
    "ana.flores78@outlook.com",      # bien
    "luis_jimenez20@gmail.com",      # bien
    "renatareyes62@@outlook.com",    # dos arrobas
    "paula.rodriguez17@",            # sin dominio
    "diego.ramirez11@hotmail",       # sin extension
    "mariarodriguez86yahoo.com",     # sin arroba
    "juan.hernandez65 @ live.com",   # espacios alrededor de la arroba
    "",                              # vacio
]

telefonos = [
    "80-4512-0612",       # el formato "oficial"
    "+52 43-7814-0090",   # con lada de pais
    "(24) 7544-5493",     # con parentesis
    "15 3019 3755",       # separado por espacios
    "57.4604.7817",         # todo junto
    "96-8256-8",          # incompleto: solo 7 digitos
    "",                   # vacio
    "5533672345",
    "55a2456b2345",
    "235293401234",
    "123-4567.8901",
    "1234-4567.8901",
]

fechas = [
    "2024-12-08 02:13:41",      # ISO, el formato "oficial"
    "20240118",                 # sin separadores
    "05-23-2024",               # mes primero
    "28-09-24 03:15:05",        # anio de dos digitos
    "16/08/2023 02:06",         # con diagonales y sin segundos
    "28 de febrero de 2024",    # en palabras
    "",                         # vacio
]

print("correos  :", len(correos))
print("telefonos:", len(telefonos))
print("fechas   :", len(fechas))

correos  : 8
telefonos: 12
fechas   : 7


## Sesión 1 — Metacaracteres, clases y anclas

Una regex es una cadena de texto que se lee como una **descripción**. El patrón
`r"\d{2}-\d{4}-\d{4}"` se lee "dos dígitos, un guion, cuatro dígitos, un guion,
cuatro dígitos". Todo lo de hoy es vocabulario para escribir descripciones así.

Para ir probando necesitamos una sola función: `re.fullmatch(patron, cadena)`,
que devuelve un objeto `Match` si **toda** la cadena cumple el patrón, y `None`
si no. La familia completa va mañana.


In [22]:
print(re.fullmatch(r"\d{2}-\d{4}-\d{4}", "80-4512-0612"))

<re.Match object; span=(0, 12), match='80-4512-0612'>


In [23]:
print(re.fullmatch(r"\d{2}-\d{4}-\d{4}", "96-8256-8"))

None


In [24]:
for telefono in telefonos:
    print(telefono, re.fullmatch(r"\d{2,3}.\d{4}.\d{4}", telefono))

80-4512-0612 <re.Match object; span=(0, 12), match='80-4512-0612'>
+52 43-7814-0090 None
(24) 7544-5493 None
15 3019 3755 <re.Match object; span=(0, 12), match='15 3019 3755'>
57.4604.7817 <re.Match object; span=(0, 12), match='57.4604.7817'>
96-8256-8 None
 None
5533672345 None
55a2456b2345 <re.Match object; span=(0, 12), match='55a2456b2345'>
235293401234 <re.Match object; span=(0, 12), match='235293401234'>
123-4567.8901 <re.Match object; span=(0, 13), match='123-4567.8901'>
1234-4567.8901 None


### Siempre con `r"..."`

Te habrás fijado en la `r` antes de las comillas. Es una **cadena cruda** (*raw
string*) y le dice a Python que no interprete las barras invertidas. Hace falta
porque la barra invertida es el carácter de escape de **dos** lenguajes a la vez:
el de Python y el de las regex. Sin la `r`, Python se come la barra antes de que
el módulo `re` llegue a verla.

El ejemplo clásico es `\b`, que en una regex significa "frontera de palabra" y en
una cadena normal de Python significa "retroceso" (el carácter número 8):


In [3]:
print("con r:", repr(r"\bgol\b"))
print("sin r:", repr("\bgol\b"))
print()
print("longitud con r:", len(r"\bgol\b"), "caracteres")
print("longitud sin r:", len("\bgol\b"), "caracteres  <- Python ya se comio las barras")

con r: '\\bgol\\b'
sin r: '\x08gol\x08'

longitud con r: 7 caracteres
longitud sin r: 5 caracteres  <- Python ya se comio las barras


In [5]:
texto = "gol de media cancha, golazo de Messi, autogol"

print("con r:", re.findall(r"\bgol\b", texto))
print("sin r:", re.findall("\bgol\b", texto), "<- no encuentra nada")

con r: ['gol']
sin r: [] <- no encuentra nada


> **Regla sin excepciones: toda regex se escribe con `r"..."`.** Aunque el patrón
> no tenga barras invertidas, póntela. Cuesta un carácter y te ahorra la clase de
> error que no da ningún mensaje: simplemente no encuentra nada, y tú reportas
> cero coincidencias creyendo que tus datos están limpios.


### Los metacaracteres

Casi todos los caracteres de un patrón se buscan tal cual: la regex `r"gol"`
encuentra la palabra "gol". Los que **no** se buscan tal cual, porque tienen un
significado especial, son catorce —once significados, porque tres de ellos
vienen en pareja:

| Carácter | Significa | Ejemplo |
|---|---|---|
| `.` | cualquier carácter, menos el salto de línea | `r"g.l"` acepta `gol`, `gal` y `g7l` |
| `^` | principio de la cadena; dentro de `[...]`, niega la clase | `r"^80"` exige empezar con `80` |
| `$` | final de la cadena | `r"com$"` exige terminar en `com` |
| `*` | cero o más veces lo anterior | `r"\d*"` acepta `""`, `7` y `123` |
| `+` | **una** o más veces lo anterior | `r"\d+"` acepta `7` y `123`, pero no `""` |
| `?` | cero o una vez, o sea opcional; detrás de otro cuantificador lo vuelve perezoso | `r"-?\d+"` acepta `12` y `-12` |
| `{` `}` | cuántas repeticiones exactamente | `r"\d{4}"` exige cuatro dígitos |
| `[` `]` | una clase: **uno** de los caracteres de adentro | `r"[aeiou]"` acepta una vocal |
| `\` | escapa el siguiente carácter, o inicia una abreviatura | `r"\."` es un punto; `r"\d"` es un dígito |
| `\|` | una alternativa o la otra | `r"(?:gol\|falta)"` acepta `gol` o `falta` |
| `(` `)` | agrupa, y captura lo agrupado | `r"(\d{2})-(\d{4})"` captura dos grupos |

No hay que memorizarla hoy: cada renglón tiene su sección más adelante, y al
final del notebook está la chuleta. Lo que sí conviene que se te quede es
**cuáles son**, porque son los catorce caracteres que no puedes teclear a la
ligera dentro de un patrón.

Si necesitas buscar uno de ellos **literalmente**, lo escapas con una barra
invertida. El caso que te va a tocar sin falta es el punto de un correo:


In [6]:
# El punto sin escapar significa "cualquier caracter"
print(re.fullmatch(r"gmail.com", "gmail.com"))
print(re.fullmatch(r"gmail.com", "gmailXcom"), "<- lo acepta, y no deberia")
print()
# Escapado, es un punto de verdad
print(re.fullmatch(r"gmail\.com", "gmail.com"))
print(re.fullmatch(r"gmail\.com", "gmailXcom"), "<- ahora si lo rechaza")

<re.Match object; span=(0, 9), match='gmail.com'>
<re.Match object; span=(0, 9), match='gmailXcom'> <- lo acepta, y no deberia

<re.Match object; span=(0, 9), match='gmail.com'>
None <- ahora si lo rechaza


### Clases de caracteres: `[...]`

Entre corchetes pones un **conjunto** de caracteres, y la clase cumple con
cualquiera de ellos —**uno solo**, no todos. Un guion dentro de los corchetes
significa "rango", y un `^` al principio de los corchetes significa "cualquiera
**menos** estos".

| Clase | Cumple con |
|---|---|
| `[aeiou]` | una vocal minúscula |
| `[a-z]` | una letra minúscula |
| `[A-Za-z]` | una letra, de cualquier caja |
| `[0-9]` | un dígito |
| `[A-Za-z0-9._]` | letra, dígito, punto o guion bajo |
| `[^0-9]` | cualquier cosa que **no** sea un dígito |

> Dentro de los corchetes los metacaracteres pierden su poder: `[.]` y `\.` son
> lo mismo. Las dos excepciones son `^` (si va en primer lugar) y `-` (si va en
> medio de dos caracteres). Para que el guion sea literal, ponlo al final.


In [6]:
print(re.fullmatch(r"[A-Za-z]", "g"))
print(re.fullmatch(r"[A-Za-z]", "7"))
print()
# Un guion al final de los corchetes es un guion literal, no un rango
print("como rango  :", re.findall(r"[a-c]", "abcdef"))
print("como literal:", re.findall(r"[ac-]", "a-b-c"))

<re.Match object; span=(0, 1), match='g'>
None

como rango  : ['a', 'b', 'c']
como literal: ['a', '-', '-', 'c']


### Las abreviaturas: `\d`, `\w`, `\s`

Tres clases son tan comunes que tienen abreviatura. Cada una tiene su versión en
mayúscula, que significa exactamente lo contrario.

| Abreviatura | Equivale a | Cumple con |
|---|---|---|
| `\d` | `[0-9]` | un dígito |
| `\D` | `[^0-9]` | lo que no es dígito |
| `\w` | `[A-Za-z0-9_]` | letra, dígito o guion bajo |
| `\W` | `[^A-Za-z0-9_]` | lo que no es eso |
| `\s` | espacio, tabulador, salto de línea | un espacio en blanco |
| `\S` | lo contrario de `\s` | lo que no es espacio |

> **Ojo con `\w`:** incluye el guion bajo, pero **no** el punto ni el guion
> normal. Y en Python 3 incluye además letras acentuadas y `ñ`, porque las
> cadenas son Unicode —útil para la columna `direccion`, y sorprendente si vienes
> de otro lenguaje. Con la bandera `re.ASCII` lo limitas a `[A-Za-z0-9_]`.


In [7]:
print(re.fullmatch(r"\w+", "jose_gonzalez33"))
print(re.fullmatch(r"\w+", "jose.gonzalez33"), "<- el punto no es \\w")
print()
print("una palabra  :", re.fullmatch(r"\w+", "Monterrey"))
print("con espacios :", re.fullmatch(r"\w+", "Nuevo Leon"), "<- el espacio tampoco es \\w")
print("con acento   :", re.fullmatch(r"\w+", "Querétaro"), "<- \\w si acepta acentos")
print("solo ASCII   :", re.fullmatch(r"\w+", "Querétaro", re.ASCII), "<- con re.ASCII ya no")

<re.Match object; span=(0, 15), match='jose_gonzalez33'>
None <- el punto no es \w

una palabra  : <re.Match object; span=(0, 9), match='Monterrey'>
con espacios : None <- el espacio tampoco es \w
con acento   : <re.Match object; span=(0, 9), match='Querétaro'> <- \w si acepta acentos
solo ASCII   : None <- con re.ASCII ya no


### Cuantificadores: cuántas veces

Un cuantificador va **después** de lo que repite y dice cuántas repeticiones
acepta.

| Cuantificador | Significa |
|---|---|
| `a*` | cero o más `a` |
| `a+` | **una** o más `a` |
| `a?` | cero o una `a` — o sea, opcional |
| `a{3}` | exactamente tres |
| `a{2,4}` | entre dos y cuatro |
| `a{2,}` | dos o más |

La diferencia entre `*` y `+` es la que más cuesta: `*` acepta la cadena vacía y
`+` no. En una validación casi siempre quieres `+`, porque tus columnas tienen
celdas vacías y no te sirve que pasen como buenas.


In [8]:
print("\\d* contra cadena vacia:", re.fullmatch(r"\d*", ""), "<- la acepta")
print("\\d+ contra cadena vacia:", re.fullmatch(r"\d+", ""), "<- la rechaza, que es lo que queremos")

\d* contra cadena vacia: <re.Match object; span=(0, 0), match=''> <- la acepta
\d+ contra cadena vacia: None <- la rechaza, que es lo que queremos


In [9]:
# El formato oficial de telefono: 2 digitos, guion, 4, guion, 4
patron_tel = r"\d{2}-\d{4}-\d{4}"

for t in telefonos:
    resultado = re.fullmatch(patron_tel, t)
    print("%-22s -> %s" % (repr(t), "OK" if resultado else "mal formado"))

'80-4512-0612'         -> OK
'+52 43-7814-0090'     -> mal formado
'(24) 7544-5493'       -> mal formado
'15 3019 3755'         -> mal formado
'5746047817'           -> mal formado
'96-8256-8'            -> mal formado
''                     -> mal formado


Seis de los siete quedaron marcados como mal formados, y solo dos lo están de
verdad: el de 7 dígitos y el vacío. Los otros cuatro son el **mismo teléfono**
escrito de otra manera, y tu patrón los acaba de tirar a la basura junto con los
que sí están roídos.

Guárdate esa distinción —*inválido* no es lo mismo que *mal escrito pero
recuperable*— porque es el centro de la sesión 3.


### Anclas: `^`, `$` y `\b`

Una ancla no consume caracteres: solo exige que la posición sea la correcta.

| Ancla | Exige estar |
|---|---|
| `^` | al principio de la cadena |
| `$` | al final de la cadena |
| `\b` | en una frontera de palabra |

Hacen falta porque `re.search` busca el patrón **en cualquier parte** de la
cadena, y ahí está la trampa de validación más común de la semana:


In [10]:
sucio = "el telefono es 80-4512-0612, apuntalo"

print("search   :", re.search(r"\d{2}-\d{4}-\d{4}", sucio))
print("fullmatch:", re.fullmatch(r"\d{2}-\d{4}-\d{4}", sucio))

search   : <re.Match object; span=(15, 27), match='80-4512-0612'>
fullmatch: None


In [11]:
# El caso que de verdad te va a morder: un valor con basura pegada
print("search    :", re.search(r"\d{2}-\d{4}-\d{4}", "80-4512-06123456"))
print("fullmatch :", re.fullmatch(r"\d{2}-\d{4}-\d{4}", "80-4512-06123456"))
print()
print("search con anclas:", re.search(r"^\d{2}-\d{4}-\d{4}$", "80-4512-06123456"))

search    : <re.Match object; span=(0, 12), match='80-4512-0612'>
fullmatch : None

search con anclas: None


Las dos formas de validar son equivalentes:

| Forma | Patrón |
|---|---|
| `re.fullmatch(patron, s)` | sin anclas |
| `re.search(r"^" + patron + r"$", s)` | con `^` y `$` |

Usa la que te parezca más clara, pero **usa una**. `re.search` con un patrón sin
anclas no valida: encuentra. Un `telefono` de 16 dígitos te pasaría como bueno, y
tu reporte de la P4 diría que la columna está más limpia de lo que está.

> **Y `re.match`, que no usamos en este curso**, ancla solo el principio: acepta
> que sobre texto al final. Es una fuente silenciosa de falsos positivos. Aquí va
> `fullmatch` para validar y `search` para buscar.


In [12]:
frase = "El Col. Vista Hermosa queda en Colima, no en Col. Centro"

print("sin \\b:", re.findall(r"Col", frase), "<- tres: cuenta el pedazo de 'Colima'")
print("con \\b:", re.findall(r"\bCol\b", frase), "<- dos: solo la palabra completa")

sin \b: ['Col', 'Col', 'Col'] <- tres: cuenta el pedazo de 'Colima'
con \b: ['Col', 'Col'] <- dos: solo la palabra completa


### Ejercicios sesión 1

Ninguno está resuelto. Del 1 al 4 son de calentamiento; el 5 ya es sobre **tu**
dataset.


In [13]:
# Ejercicio 1: Escribe el patron que acepte exactamente 8 digitos seguidos
# (el formato "20240118" de la lista `fechas`). Pruebalo con fullmatch contra
# los 7 valores de `fechas` e imprime cuales acepta.

# Tu codigo aqui:

In [14]:
# Ejercicio 2: Comprueba que r"\d{2}-\d{4}-\d{4}" acepta "80-4512-0612" pero
# rechaza "8-4512-0612". Despues escribe una variante que acepte 2 O 3 digitos
# en el primer grupo, y verifica que ahora acepta los dos.

# Tu codigo aqui:

In [15]:
# Ejercicio 3: Con una clase negada, encuentra todos los caracteres de
#   "Calle Madero #1357, Col. Vista Hermosa, Tijuana"
# que NO son letras, digitos ni espacios. Imprime la lista de caracteres
# distintos que te salio (pista: set()).

# Tu codigo aqui:

In [16]:
# Ejercicio 4: Escribe un patron que acepte un codigo postal mexicano: cinco
# digitos exactos. Pruebalo contra "01000", "1000", "010000" y "0100a".
# Cuantos de los cuatro deberia aceptar?

# Tu codigo aqui:

In [17]:
# Ejercicio 5: Carga la columna `email` de tu datos/{tema}-ruido.csv (el patron
# de la semana 6: open, split("|"), localizar la columna por nombre con .index())
# y cuenta:
#   - cuantos valores estan vacios
#   - de los no vacios, cuantos pasan re.fullmatch(r"\S+@\S+\.\S+", valor)
# Ese patron es malisimo a proposito. El viernes vas a ver exactamente por que.

# Tu codigo aqui:

## Sesión 2 — Grupos de captura, `search`, `findall`, `sub`

Ayer solo preguntabas sí o no. Hoy vas a **sacar** las partes de un valor y a
**reescribirlas**, que es lo que hace falta cuando el dato está mal formado pero
recuperable —la mayoría de los casos de tu dataset.


### `re.search` devuelve un objeto, no una cadena

Ya viste que `re.search` imprime algo como `<re.Match object; span=(15, 27),
match='80-4512-0612'>`. Ese objeto tiene lo que necesitas:

| Método | Te da |
|---|---|
| `.group()` o `.group(0)` | el texto completo que coincidió |
| `.start()`, `.end()` | las posiciones |
| `.span()` | la tupla `(inicio, fin)` |

Y si no hubo coincidencia devuelve `None`, que **no tiene** esos métodos. De ahí
el error más frecuente de hoy:


In [18]:
m = re.search(r"\d{2}-\d{4}-\d{4}", "el telefono es 80-4512-0612, apuntalo")

print("objeto  :", m)
print("group() :", m.group())
print("span()  :", m.span())
print("recorte :", "el telefono es 80-4512-0612, apuntalo"[m.start():m.end()])

objeto  : <re.Match object; span=(15, 27), match='80-4512-0612'>
group() : 80-4512-0612
span()  : (15, 27)
recorte : 80-4512-0612


In [19]:
try:
    re.search(r"\d{2}-\d{4}-\d{4}", "no hay telefono aqui").group()
except AttributeError as e:
    print("AttributeError:", e)

AttributeError: 'NoneType' object has no attribute 'group'


> **`None` no tiene `.group()`.** Revisa siempre antes de usarlo:
>
> ```python
> m = re.search(patron, valor)
> if m:
>     usar(m.group())
> ```
>
> Con 100,000 filas te va a tocar el `None` seguro, aunque en tus diez pruebas no
> haya salido.


### Grupos de captura: `(...)`

Un paréntesis marca un pedazo del patrón para **recuperarlo aparte**. Se numeran
de izquierda a derecha empezando en 1; el grupo 0 es siempre la coincidencia
completa.


In [20]:
m = re.fullmatch(r"(\d{2})-(\d{4})-(\d{4})", "80-4512-0612")

print("group(0):", m.group(0))
print("group(1):", m.group(1))
print("group(2):", m.group(2))
print("group(3):", m.group(3))
print()
print("groups():", m.groups(), "<- una tupla, sin el 0")

group(0): 80-4512-0612
group(1): 80
group(2): 4512
group(3): 0612

groups(): ('80', '4512', '0612') <- una tupla, sin el 0


### Grupos con nombre: `(?P<nombre>...)`

Contar paréntesis para saber si el año es el grupo 1 o el 3 es exactamente la
clase de cosa que se rompe cuando editas el patrón. Ponles nombre:


In [21]:
patron_iso = r"(?P<anio>\d{4})-(?P<mes>\d{2})-(?P<dia>\d{2})"
m = re.fullmatch(patron_iso, "2024-12-08")

print("anio:", m.group("anio"))
print("mes :", m.group("mes"))
print("dia :", m.group("dia"))
print()
print("todos de golpe:", m.groupdict(), "<- un diccionario")

anio: 2024
mes : 12
dia : 08

todos de golpe: {'anio': '2024', 'mes': '12', 'dia': '08'} <- un diccionario


### Alternancia `|` y grupos sin captura `(?:...)`

La barra vertical significa "o". Casi siempre necesita un paréntesis alrededor
para delimitar **hasta dónde** llega la alternativa, porque `|` tiene la
precedencia más baja de todo el lenguaje de las regex: sin paréntesis parte el
patrón **completo** en dos.

Si el paréntesis está solo para agrupar y no quieres que capture nada, usa
`(?:...)`. No cambia lo que el patrón acepta; cambia lo que sale en `.groups()` y
en `findall`.


In [22]:
# Sin parentesis, el | parte el patron completo en dos
print(re.fullmatch(r"gmail|hotmail\.com", "gmail"), "<- acepta 'gmail' pelon")
print(re.fullmatch(r"gmail|hotmail\.com", "gmail.com"), "<- y rechaza 'gmail.com'")
print()
# Con parentesis, solo alterna el dominio
patron = r"(?:gmail|hotmail|yahoo)\.com"
for dominio in ("gmail.com", "hotmail.com", "live.com"):
    print("%-14s -> %s" % (dominio, "OK" if re.fullmatch(patron, dominio) else "no"))

<re.Match object; span=(0, 5), match='gmail'> <- acepta 'gmail' pelon
None <- y rechaza 'gmail.com'

gmail.com      -> OK
hotmail.com    -> OK
live.com       -> no


### `re.findall`: todas las coincidencias

`findall` devuelve una **lista de cadenas**... siempre que tu patrón no tenga
grupos de captura. Si los tiene, cambia de comportamiento sin avisar: devuelve
los grupos, no la coincidencia completa. Es la trampa número uno de la semana.


In [23]:
registro = "contactos: 80-4512-0612, 52-8915-9923 y 89-6943-0962"

print("sin grupos:", re.findall(r"\d{2}-\d{4}-\d{4}", registro))
print("un grupo  :", re.findall(r"\d{2}-(\d{4})-\d{4}", registro), "<- solo el grupo")
print("dos grupos:", re.findall(r"(\d{2})-(\d{4})-\d{4}", registro), "<- tuplas")
print()
print("sin captura:", re.findall(r"\d{2}-(?:\d{4})-\d{4}", registro), "<- (?:...) lo arregla")

sin grupos: ['80-4512-0612', '52-8915-9923', '89-6943-0962']
un grupo  : ['4512', '8915', '6943'] <- solo el grupo
dos grupos: [('80', '4512'), ('52', '8915'), ('89', '6943')] <- tuplas

sin captura: ['80-4512-0612', '52-8915-9923', '89-6943-0962'] <- (?:...) lo arregla


Si lo que querías eran las coincidencias completas **y** necesitas el paréntesis
para agrupar, tienes dos salidas: cambiar `(...)` por `(?:...)`, o usar
`finditer` y pedir `m.group(0)`.


### `re.finditer`: lo mismo, pero con los objetos `Match`

Cuando además de los pedazos quieres saber **dónde** estaban, o quieres los
grupos con nombre, usa `finditer`. Devuelve un iterador de `Match`.


In [24]:
for m in re.finditer(r"(\d{2})-(\d{4})-(\d{4})", registro):
    print("en la posicion %-3d esta %s   (lada %s)" % (m.start(), m.group(0), m.group(1)))

en la posicion 11  esta 80-4512-0612   (lada 80)
en la posicion 25  esta 52-8915-9923   (lada 52)
en la posicion 40  esta 89-6943-0962   (lada 89)


### `re.sub`: reemplazar

`re.sub(patron, reemplazo, cadena)` cambia **todas** las coincidencias. El
reemplazo puede ser de tres tipos:

| Reemplazo | Para qué |
|---|---|
| una cadena fija | borrar o estandarizar |
| una cadena con `\1`, `\2` o `\g<nombre>` | reordenar usando lo capturado |
| una **función** que recibe el `Match` | cuando hay que decidir algo |

El primero es el idioma más útil de la semana: borrar todo lo que no sea dígito.


In [25]:
# Dejar solo los digitos: \D es "lo que no es digito"
for t in telefonos:
    print("%-22s -> %s" % (repr(t), re.sub(r"\D", "", t)))

'80-4512-0612'         -> 8045120612
'+52 43-7814-0090'     -> 524378140090
'(24) 7544-5493'       -> 2475445493
'15 3019 3755'         -> 1530193755
'5746047817'           -> 5746047817
'96-8256-8'            -> 9682568
''                     -> 


In [26]:
# Reordenar usando los grupos capturados por numero
print("por numero:", re.sub(r"(\d{4})-(\d{2})-(\d{2})", r"\3/\2/\1", "2024-12-08"))

# Lo mismo con nombres: mismo resultado, pero se lee y se edita sin contar parentesis
print("por nombre:", re.sub(patron_iso, r"\g<dia>/\g<mes>/\g<anio>", "2024-12-08"))

por numero: 08/12/2024
por nombre: 08/12/2024


> **El reemplazo también va con `r"..."`.** Sin la `r`, `"\1"` es el carácter
> número 1 de la tabla ASCII, no "el grupo 1".


In [27]:
# Con una funcion: enmascarar el correo dejando ver solo la primera letra
def enmascarar(m):
    usuario = m.group("usuario")
    return usuario[0] + "*" * (len(usuario) - 1) + "@" + m.group("dominio")


patron_correo = r"(?P<usuario>[\w.]+)@(?P<dominio>[\w.]+)"

for c in correos[:4]:
    print("%-32s -> %s" % (repr(c), re.sub(patron_correo, enmascarar, c)))

'ana.flores78@outlook.com'       -> a***********@outlook.com
'luis_jimenez20@gmail.com'       -> l*************@gmail.com
'renatareyes62@@outlook.com'     -> renatareyes62@@outlook.com
'paula.rodriguez17@'             -> paula.rodriguez17@


Fíjate en los dos últimos: el de las dos arrobas y el que no tiene dominio
salieron **sin cambios**. No es un error de `sub`: el patrón no encontró nada que
reemplazar, y `re.sub` en ese caso devuelve la cadena original tal cual, sin
avisar.

> **`re.sub` nunca falla.** Si no hubo coincidencias, te devuelve lo mismo que le
> diste. Si tu pipeline supone que `sub` arregló el valor, vas a tener valores
> sucios marcados como limpios. Cuando importe, compara el resultado contra la
> entrada, o valida después de reemplazar.


### `re.compile`: el patrón como objeto

Hasta ahora pasamos el patrón como cadena en cada llamada. `re.compile` lo
traduce una sola vez y te devuelve un objeto con los mismos métodos
(`.fullmatch`, `.search`, `.findall`, `.sub`), ya sin el primer argumento. Tiene
dos ventajas, y la segunda es la que importa:

1. Si vas a aplicar el patrón 100,000 veces —o sea, en tu práctica— no lo
   reinterpreta cada vez.
2. El patrón queda **con nombre y en un solo lugar**. Cuando lo tengas que
   corregir, lo corriges una vez.


In [28]:
RE_TELEFONO = re.compile(r"\d{2}-\d{4}-\d{4}")

print(RE_TELEFONO.fullmatch("80-4512-0612"))
print(RE_TELEFONO.findall(registro))
print()
print("el patron sigue a la vista:", RE_TELEFONO.pattern)

<re.Match object; span=(0, 12), match='80-4512-0612'>
['80-4512-0612', '52-8915-9923', '89-6943-0962']

el patron sigue a la vista: \d{2}-\d{4}-\d{4}


> La convención de este curso es ponerles **MAYÚSCULAS** a los patrones
> compilados, como a cualquier constante, y definirlos arriba del script. En la
> P4 te van a quedar tres o cuatro y conviene que se vean de un golpe.


### Banderas (*flags*)

Un argumento extra cambia cómo se interpreta el patrón. Las dos que te pueden
servir:

| Bandera | Qué hace |
|---|---|
| `re.IGNORECASE` (`re.I`) | no distingue mayúsculas de minúsculas |
| `re.ASCII` (`re.A`) | restringe `\w`, `\d` y `\s` a ASCII puro |


In [29]:
print(re.fullmatch(r"[a-z]+@gmail\.com", "Ana@Gmail.com"))
print(re.fullmatch(r"[a-z]+@gmail\.com", "Ana@Gmail.com", re.IGNORECASE))

None
<re.Match object; span=(0, 13), match='Ana@Gmail.com'>


### Cuantificadores codiciosos: `.*` contra `.*?`

Por omisión un cuantificador agarra **todo lo que puede** y solo suelta si el
resto del patrón no cumple. A eso se le llama *codicioso* (*greedy*). Si le pones
un `?` detrás se vuelve *perezoso* (*lazy*) y agarra lo mínimo.


In [30]:
linea = "Col. Vista Hermosa, Col. Roma Norte, Tijuana"

print("codicioso:", re.findall(r"Col\..*,", linea))
print("perezoso :", re.findall(r"Col\..*?,", linea))

codicioso: ['Col. Vista Hermosa, Col. Roma Norte,']
perezoso : ['Col. Vista Hermosa,', 'Col. Roma Norte,']


El codicioso se lleva hasta la **última** coma de la línea; el perezoso se para
en la primera. Ninguno de los dos está mal: depende de qué querías. Lo que está
mal es no saber cuál usaste.

> **Y la regla de fondo:** `.*` es el comodín que todo lo acepta, y por eso casi
> nunca es lo que quieres en una **validación**. Si puedes decir qué caracteres
> esperas (`\d`, `[\w.]`, `[^@]`), dilo. Mañana vas a ver qué tan malo es un
> validador hecho con comodines.


### Ejercicios sesión 2

Del 6 al 8 son sobre las listas de este notebook; el 9 y el 10 ya son sobre tu
dataset.


In [31]:
# Ejercicio 6: Sobre `registro`, usa finditer con grupos CON NOMBRE (lada,
# medio, final) e imprime un diccionario por cada telefono encontrado.

# Tu codigo aqui:

In [32]:
# Ejercicio 7: Con re.sub y una funcion de reemplazo, convierte
#   "28-09-24 03:15:05"  ->  "2024-09-28 03:15:05"
# Asume que el anio de dos digitos es 20xx. Verifica que tu codigo no toca una
# fecha que ya viene en ISO.

# Tu codigo aqui:

In [33]:
# Ejercicio 8: Predice que devuelve
#   re.findall(r"(\w+)@(\w+)\.com", "a@gmail.com y b@yahoo.com")
# antes de correrlo. Despues corrigelo para que devuelva los dos correos
# COMPLETOS sin dejar de usar parentesis.

# Tu codigo aqui:

In [34]:
# Ejercicio 9: Sobre tu columna `telefono` completa, cuenta:
#   (a) cuantos valores cumplen fullmatch con r"\d{2}-\d{4}-\d{4}"
#   (b) cuantos tienen exactamente 10 digitos despues de re.sub(r"\D", "", v)
# Los dos numeros NO son iguales. Explica en un comentario a que se debe la
# diferencia y cual de los dos te conviene reportar.

# Tu codigo aqui:

In [35]:
# Ejercicio 10: Compila un RE_EMAIL y escribe
#   es_email_valido(valor)
# que devuelva True o False y que maneje la cadena vacia sin tronar. Correlo
# sobre tu columna `email` y reporta cuantos validos te dio. El viernes vas a
# mejorar este patron, asi que guarda el numero para comparar.

# Tu codigo aqui:

## Sesión 3 — Taller: validar correos, extraer teléfonos, normalizar fechas

Las tres columnas de texto de tu dataset, en orden de dificultad. El patrón de
trabajo es el mismo las tres veces, y es lo que hay que aprenderse:

1. **Mira los datos** y haz el catálogo de las formas que aparecen: la buena y
   todas las malas.
2. **Escribe el patrón** para la forma buena.
3. **Pruébalo contra las malas.** Si las acepta, el patrón está mal —no los
   datos.

El paso 3 es el que todo el mundo se salta, y es el único que te dice si tu
validador sirve.


### Correos: por qué `\S+@\S+\.\S+` no sirve

Ese patrón —"algo, arroba, algo, punto, algo"— es el que sale en el primer
resultado de cualquier búsqueda, y es el que usaste en el ejercicio 5. Vamos a
correrlo contra el catálogo:


In [36]:
malo = r"\S+@\S+\.\S+"

for c in correos:
    print("%-32s -> %s" % (repr(c), "OK" if re.fullmatch(malo, c) else "mal formado"))

'ana.flores78@outlook.com'       -> OK
'luis_jimenez20@gmail.com'       -> OK
'renatareyes62@@outlook.com'     -> OK
'paula.rodriguez17@'             -> mal formado
'diego.ramirez11@hotmail'        -> mal formado
'mariarodriguez86yahoo.com'      -> mal formado
'juan.hernandez65 @ live.com'    -> mal formado
''                               -> mal formado


Acertó en siete de ocho, pero dejó pasar el de las **dos arrobas**. La razón es
que `\S` significa "cualquier cosa que no sea espacio", y eso incluye la arroba:
la segunda `\S+` se tragó la segunda arroba sin protestar.

Y con casos un poco más hostiles se ve mejor de qué tamaño es el agujero:


In [37]:
casos_frontera = [
    "ana@correo.com.mx",            # valido: dominio con dos puntos
    "renatareyes62@@outlook.com",   # dos arrobas
    "$$$@###.!!!",                  # pura basura
    "ana@outlook.",                 # termina en punto
    "@outlook.com",                 # sin usuario
    "ana@.com",                     # sin dominio
]

for c in casos_frontera:
    print("%-32s -> %s" % (repr(c), "OK" if re.fullmatch(malo, c) else "mal formado"))

'ana@correo.com.mx'              -> OK
'renatareyes62@@outlook.com'     -> OK
'$$$@###.!!!'                    -> OK
'ana@outlook.'                   -> mal formado
'@outlook.com'                   -> mal formado
'ana@.com'                       -> mal formado


`$$$@###.!!!` pasó como correo válido. Un validador que acepta eso no te está
diciendo nada sobre la calidad de tu columna.

Vamos a describir la forma de verdad, pedazo por pedazo:

| Pedazo | Patrón | Por qué |
|---|---|---|
| usuario | `[A-Za-z0-9._%+-]+` | letras, dígitos y los signos que un correo admite — **sin arroba y sin espacios** |
| arroba | `@` | exactamente una |
| dominio | `[A-Za-z0-9-]+` | la parte antes del primer punto |
| subdominios | `(?:\.[A-Za-z0-9-]+)*` | cero o más `.algo`, para que `com.mx` siga siendo válido |
| extensión | `\.[A-Za-z]{2,}` | punto y al menos dos letras, **solo letras** |

La clave de todas está en lo que **no** incluyen: ninguna clase admite `@` ni
espacio, así que las dos arrobas y los espacios alrededor ya no tienen por dónde
pasar. Y como la extensión exige `[A-Za-z]{2,}`, `###.!!!` tampoco.


In [38]:
# Python pega cadenas consecutivas, asi que el patron se puede comentar por partes
RE_EMAIL = re.compile(
    r"[A-Za-z0-9._%+-]+"        # usuario
    r"@"                        # la arroba, exactamente una
    r"[A-Za-z0-9-]+"            # dominio
    r"(?:\.[A-Za-z0-9-]+)*"     # subdominios opcionales: .com.mx
    r"\.[A-Za-z]{2,}"           # extension: punto y al menos dos letras
)

print(RE_EMAIL.pattern)

[A-Za-z0-9._%+-]+@[A-Za-z0-9-]+(?:\.[A-Za-z0-9-]+)*\.[A-Za-z]{2,}


In [39]:
print("--- el catalogo ---")
for c in correos:
    print("%-32s -> %s" % (repr(c), "OK" if RE_EMAIL.fullmatch(c) else "mal formado"))

print()
print("--- los casos frontera ---")
for c in casos_frontera:
    print("%-32s -> %s" % (repr(c), "OK" if RE_EMAIL.fullmatch(c) else "mal formado"))

--- el catalogo ---
'ana.flores78@outlook.com'       -> OK
'luis_jimenez20@gmail.com'       -> OK
'renatareyes62@@outlook.com'     -> mal formado
'paula.rodriguez17@'             -> mal formado
'diego.ramirez11@hotmail'        -> mal formado
'mariarodriguez86yahoo.com'      -> mal formado
'juan.hernandez65 @ live.com'    -> mal formado
''                               -> mal formado

--- los casos frontera ---
'ana@correo.com.mx'              -> OK
'renatareyes62@@outlook.com'     -> mal formado
'$$$@###.!!!'                    -> mal formado
'ana@outlook.'                   -> mal formado
'@outlook.com'                   -> mal formado
'ana@.com'                       -> mal formado


Ocho de ocho y seis de seis, incluyendo el `com.mx`, que es válido y hay que
aceptar.

> **Ningún patrón de una línea valida correos de verdad.** La especificación
> oficial (RFC 5322) admite cosas que nadie escribe, y la única prueba
> concluyente de que un correo existe es mandarle un mensaje. Lo que necesitas
> para la P4 no es perfección: es un criterio **explícito y reproducible** que
> rechace los errores que tu dataset sí tiene y que puedas defender por escrito.
> El de arriba lo es. Anótalo en tu README junto con qué decidiste aceptar.


### Teléfonos: limpiar primero, validar después

Tu columna `telefono` trae el mismo número escrito de cinco maneras. Se puede
armar un solo patrón que acepte las cinco, y queda así:


In [40]:
monstruo = re.compile(r"(?:\+\d{2}\s)?(?:\(\d{2}\)|\d{2})[\s-]?\d{4}[\s-]?\d{4}")

for t in telefonos:
    print("%-22s -> %s" % (repr(t), "OK" if monstruo.fullmatch(t) else "mal formado"))

'80-4512-0612'         -> OK
'+52 43-7814-0090'     -> OK
'(24) 7544-5493'       -> OK
'15 3019 3755'         -> OK
'5746047817'           -> OK
'96-8256-8'            -> mal formado
''                     -> mal formado


Funciona —los siete quedan bien clasificados— y en seis meses nadie, tú incluido,
va a poder leerlo. Pero el problema de fondo no es que sea ilegible: es que solo
sabe decir *sí* o *no*. Cuando tengas que explicar en tu reporte **por qué**
rechazaste 1,884 teléfonos, ese patrón no te sirve de nada.

La alternativa es en dos pasos, y es la que recomiendo:

1. **Normaliza:** borra todo lo que no sea dígito con `re.sub(r"\D", "", v)`.
2. **Valida el resultado contando dígitos:** 10 es válido; 12 que empiezan con
   `52` es válido con lada de país y se le quita; cualquier otra cosa, no.

Con dos pasos puedes decir "tenía 9 dígitos". Con el monstruo solo puedes decir
"no cumplió".


In [41]:
def normalizar_telefono(valor):
    # Devuelve la tupla (telefono_formateado, motivo). Si el valor es
    # recuperable, motivo es None; si no, el formateado es None y el motivo
    # dice que paso.
    digitos = re.sub(r"\D", "", valor)

    if digitos == "":
        return None, "vacio"
    if len(digitos) == 12 and digitos.startswith("52"):
        digitos = digitos[2:]                 # le quitamos la lada de pais
    if len(digitos) != 10:
        return None, "tiene %d digitos, se esperaban 10" % len(digitos)

    return "%s-%s-%s" % (digitos[:2], digitos[2:6], digitos[6:]), None


for t in telefonos:
    formateado, motivo = normalizar_telefono(t)
    print("%-22s -> %-16s %s" % (repr(t), formateado, motivo or ""))

'80-4512-0612'         -> 80-4512-0612     
'+52 43-7814-0090'     -> 43-7814-0090     
'(24) 7544-5493'       -> 24-7544-5493     
'15 3019 3755'         -> 15-3019-3755     
'5746047817'           -> 57-4604-7817     
'96-8256-8'            -> None             tiene 7 digitos, se esperaban 10
''                     -> None             vacio


Cinco de los siete se recuperaron y quedaron en el mismo formato, listos para
comparar entre sí. Los dos que no, vienen con la razón escrita.

> **Ese segundo valor de retorno es la mitad del trabajo de la P4.** "El 4.1% de
> los teléfonos está mal formado" es un dato. "El 1.9% está vacío y el 1.8% tiene
> 7 dígitos, lo que sugiere que se truncaron al capturarlos" es un hallazgo.


### Fechas: regex te dice la forma, no el significado

Tu columna `timestamp` trae seis formas distintas. Aquí el patrón sí vale la pena
—en lugar de limpiar y contar— porque además de validar necesitas **extraer** el
año, el mes y el día para reescribirlos.

La estrategia es una **lista de formatos**: un patrón por forma, con los mismos
tres grupos con nombre, probados en orden hasta que uno cumpla.


In [42]:
MESES = {
    "enero": 1, "febrero": 2, "marzo": 3, "abril": 4, "mayo": 5, "junio": 6,
    "julio": 7, "agosto": 8, "septiembre": 9, "octubre": 10, "noviembre": 11,
    "diciembre": 12,
}

FORMATOS = [
    ("iso",      re.compile(r"(?P<anio>\d{4})-(?P<mes>\d{2})-(?P<dia>\d{2})(?: \d{2}:\d{2}:\d{2})?")),
    ("compacto", re.compile(r"(?P<anio>\d{4})(?P<mes>\d{2})(?P<dia>\d{2})")),
    ("mes_dia",  re.compile(r"(?P<mes>\d{2})-(?P<dia>\d{2})-(?P<anio>\d{4})")),
    ("corto",    re.compile(r"(?P<dia>\d{2})-(?P<mes>\d{2})-(?P<anio>\d{2}) \d{2}:\d{2}:\d{2}")),
    ("diagonal", re.compile(r"(?P<dia>\d{2})/(?P<mes>\d{2})/(?P<anio>\d{4})(?: \d{2}:\d{2})?")),
    ("palabras", re.compile(r"(?P<dia>\d{1,2}) de (?P<mes>[a-z]+) de (?P<anio>\d{4})")),
]

for f in fechas:
    reconocido = [nombre for nombre, rx in FORMATOS if rx.fullmatch(f)]
    print("%-28s -> %s" % (repr(f), reconocido or "ninguno"))

'2024-12-08 02:13:41'        -> ['iso']
'20240118'                   -> ['compacto']
'05-23-2024'                 -> ['mes_dia']
'28-09-24 03:15:05'          -> ['corto']
'16/08/2023 02:06'           -> ['diagonal']
'28 de febrero de 2024'      -> ['palabras']
''                           -> ninguno


Cada valor cayó en **exactamente un** formato, y el vacío en ninguno. Eso no pasa
por suerte: pasa porque los patrones están anclados con `fullmatch` y porque cada
uno exige su separador. Si los hubieras escrito con `search`, `compacto` habría
cazado pedazos de la fecha ISO.

Con los grupos ya nombrados igual en los seis patrones, la conversión es una sola
función:


In [43]:
def a_iso(valor):
    # Devuelve (fecha_iso, formato_detectado), o (None, motivo) si no cuadra.
    for nombre, rx in FORMATOS:
        m = rx.fullmatch(valor)
        if not m:
            continue
        partes = m.groupdict()
        anio = int(partes["anio"])
        if anio < 100:                        # el "24" de 28-09-24
            anio += 2000
        mes = partes["mes"]
        mes = MESES[mes] if mes in MESES else int(mes)
        return "%04d-%02d-%02d" % (anio, mes, int(partes["dia"])), nombre
    return None, "formato desconocido"


for f in fechas:
    iso, nota = a_iso(f)
    print("%-28s -> %-12s (%s)" % (repr(f), iso, nota))

'2024-12-08 02:13:41'        -> 2024-12-08   (iso)
'20240118'                   -> 2024-01-18   (compacto)
'05-23-2024'                 -> 2024-05-23   (mes_dia)
'28-09-24 03:15:05'          -> 2024-09-28   (corto)
'16/08/2023 02:06'           -> 2023-08-16   (diagonal)
'28 de febrero de 2024'      -> 2024-02-28   (palabras)
''                           -> None         (formato desconocido)


### Lo que regex no puede resolver

Mira estos dos valores. Los dos están en tu dataset y los dos tienen la forma
`\d{2}-\d{2}-\d{4}`:

| Valor | Lectura 1 | Lectura 2 |
|---|---|---|
| `05-23-2024` | 23 de mayo (mes-día) | el mes 23 no existe |
| `10-09-2024` | 9 de octubre (mes-día) | 10 de septiembre (día-mes) |

El primero se resuelve solo: no hay mes 23, así que solo una lectura es posible.
El segundo **no tiene solución**: las dos lecturas son válidas y el dato no trae
nada que te permita decidir.


In [44]:
patron_mes_dia = FORMATOS[2][1]

for valor in ("05-23-2024", "10-09-2024"):
    m = patron_mes_dia.fullmatch(valor)
    primero, segundo = int(m.group("mes")), int(m.group("dia"))
    print("%s -> primer numero %2d, segundo %2d  |  %s" % (
        valor, primero, segundo,
        "solo cabe mes-dia" if primero > 12 or segundo > 12 else "AMBIGUO"))

05-23-2024 -> primer numero  5, segundo 23  |  solo cabe mes-dia
10-09-2024 -> primer numero 10, segundo  9  |  AMBIGUO


Lo correcto ahí no es apretar el patrón: es **tomar una decisión, dejarla escrita
y poder contar a cuántas filas afectó**. Por ejemplo:

> "Los N valores con forma `\d{2}-\d{2}-\d{4}` se interpretan como mes-día-año,
> porque M de ellos tienen un primer número mayor a 12 —imposible para un mes si
> fuera día-mes— y ninguno tiene un segundo número mayor a 31. Quedan K filas en
> las que las dos lecturas siguen siendo posibles."

Los números van ahí, contados por ti. Esa es la diferencia entre documentar una
decisión y suponerla.

Esa frase es análisis de datos. La regex solo te dio el material para escribirla.

> **Y la regla general de cuándo *no* usar regex:** si la estructura es anidada o
> recursiva —HTML, JSON, un CSV con comas dentro de las comillas— regex no es la
> herramienta, y no existe patrón lo bastante listo. Para eso están `json.load`,
> `csv.reader` y, a partir de la semana 9, pandas.


### Ejercicios sesión 3

Del 11 al 16 construyes, pieza por pieza, el diagnóstico de calidad de texto que
pide la P4. Ninguno está resuelto aquí.


In [45]:
# Ejercicio 11: Prueba RE_EMAIL contra 10 correos que escribas TU: cinco validos
# y cinco mal formados, ninguno de los que ya estan en este notebook. Si acepta
# uno malo o rechaza uno bueno, ajusta el patron y vuelve a correr las 10.

# Tu codigo aqui:

In [46]:
# Ejercicio 12: Sobre tu columna `email` completa, reporta cuantos valores hay
# vacios, validos y mal formados, con su porcentaje sobre el total de filas.
# Imprime los primeros 10 mal formados: puedes nombrar el tipo de error de cada
# uno? Compara el numero de validos contra el que te dio el ejercicio 10.

# Tu codigo aqui:

In [47]:
# Ejercicio 13: Escribe
#   clasificar_email(valor)
# que en lugar de True/False devuelva el MOTIVO del rechazo, uno de:
#   "valido", "vacio", "sin arroba", "arroba repetida", "con espacios",
#   "sin extension", "otro"
# Cuenta cuantos hay de cada motivo en tu columna. Pista: no es un solo patron,
# son varios probados en orden, como la lista FORMATOS de las fechas.

# Tu codigo aqui:

In [48]:
# Ejercicio 14: Corre normalizar_telefono sobre tu columna `telefono` completa y
# produce un conteo por motivo de rechazo. Responde con cifras: cuantos valores
# se recuperan SOLO con normalizar, o sea cuantos no cumplian el formato oficial
# pero si tienen 10 digitos?

# Tu codigo aqui:

In [49]:
# Ejercicio 15: Corre la lista FORMATOS sobre tu columna `timestamp` y cuenta
# cuantos valores caen en cada formato, mas los vacios y los que no cayeron en
# ninguno. Los tres grupos deben sumar el total de filas; si no suman, te falta
# catalogar un formato: encuentralo, agregalo a FORMATOS y vuelve a contar.
# Despues responde con cifras: que porcentaje de tu columna NO viene en ISO, y
# de los que tienen forma \d{2}-\d{2}-\d{4}, cuantos son genuinamente ambiguos?

# Tu codigo aqui:

In [50]:
# Ejercicio 16 (reto): Junta los tres en un solo script que lea tu CSV y escriba
# un JSON con, por cada una de las tres columnas, el conteo de validos, vacios y
# mal formados mas el desglose por motivo. Es un borrador del diagnostico de
# calidad que pide la P4 --con la diferencia de que en la P4 lo vas a escribir
# con pandas y .str, en una decima parte de las lineas.

# Tu codigo aqui:

---

## Chuleta de la semana

**Lo que describe un carácter**

| Patrón | Cumple con |
|---|---|
| `a`, `7`, `-` | ese carácter, tal cual |
| `.` | cualquier carácter (menos el salto de línea) |
| `\.` | un punto de verdad |
| `[abc]`, `[a-z]`, `[A-Za-z0-9._]` | uno de esos |
| `[^abc]` | cualquiera **menos** esos |
| `\d` `\w` `\s` | dígito, letra/dígito/`_`, espacio |
| `\D` `\W` `\S` | lo contrario de cada uno |

**Cuántas veces**

| Patrón | Repeticiones |
|---|---|
| `x*` | 0 o más |
| `x+` | 1 o más |
| `x?` | 0 o 1 (opcional) |
| `x{3}` | exactamente 3 |
| `x{2,4}` · `x{2,}` | entre 2 y 4 · 2 o más |
| `x+?` · `x*?` | lo mínimo, en vez de lo máximo |

**Dónde**

| Patrón | Exige |
|---|---|
| `^` · `$` | principio · final de la cadena |
| `\b` | frontera de palabra |

**Agrupar**

| Patrón | Para qué |
|---|---|
| `(...)` | agrupar **y** capturar — numerado desde 1 |
| `(?:...)` | agrupar sin capturar |
| `(?P<nombre>...)` | capturar con nombre |
| `a\|b` | una u otra — **siempre dentro de un paréntesis** |

**Las funciones**

| Llamada | Devuelve |
|---|---|
| `re.fullmatch(p, s)` | `Match` si **toda** `s` cumple, si no `None` |
| `re.search(p, s)` | `Match` de la primera coincidencia **en cualquier parte** |
| `re.findall(p, s)` | lista de cadenas — **de tuplas si hay grupos** |
| `re.finditer(p, s)` | iterador de `Match` |
| `re.sub(p, r, s)` | `s` con todas las coincidencias reemplazadas |
| `re.compile(p)` | el patrón como objeto, con los mismos métodos |

**Del `Match`**

| Método | Da |
|---|---|
| `.group()` / `.group(0)` | la coincidencia completa |
| `.group(1)` / `.group("mes")` | un grupo |
| `.groups()` / `.groupdict()` | todos, en tupla / en diccionario |
| `.start()` `.end()` `.span()` | las posiciones |

**Los tres idiomas que más vas a usar**

```python
re.sub(r"\D", "", valor)                  # dejar solo los digitos
re.fullmatch(RE_EMAIL, valor) is not None # validar de punta a punta
m = RX.fullmatch(valor); m.groupdict()    # extraer las partes con nombre
```


---

## Resumen de la semana

**La forma de trabajar**, que es lo que de verdad hay que llevarse:

1. Mira los datos y escribe el catálogo de formas: la buena y las malas.
2. Escribe el patrón para la buena, con clases **específicas** —no `.*` ni `\S`.
3. Pruébalo contra las malas. Si las acepta, el patrón está mal.
4. Cuando el valor es recuperable, **normaliza y luego valida**. Cuando no lo es,
   guarda el **motivo**, no solo el `False`.
5. Lo que el dato no dice —si `10-09-2024` es octubre o septiembre— no lo
   resuelve ningún patrón. Se decide, se documenta y se cuenta.

**Validar contra buscar**

| Quiero | Uso |
|---|---|
| que **toda** la cadena cumpla | `re.fullmatch(p, s)` |
| encontrar el patrón adentro | `re.search(p, s)` |
| validar con `search` | `re.search(r"^" + p + r"$", s)` |

**Las tres columnas de la P4**

| Columna | Estrategia |
|---|---|
| `email` | un patrón específico por pedazo: usuario, `@`, dominio, subdominios, extensión |
| `telefono` | `re.sub(r"\D", "", v)` y contar dígitos: 10 válido, 12 con `52` válido |
| `timestamp` | lista de formatos con los mismos grupos con nombre, probados en orden |


## Las cinco trampas de esta semana

1. **Olvidar la `r"..."`.** No da ningún error: simplemente no encuentra nada, y
   tú reportas cero coincidencias creyendo que la columna está limpia. Va también
   en el reemplazo de `re.sub`.
2. **Validar con `re.search` y un patrón sin anclas.** `search` encuentra el
   patrón en cualquier parte, así que `"80-4512-06123456"` pasa como teléfono
   bueno. Para validar, `fullmatch` o `^...$`.
3. **`.group()` sobre `None`.** Si no hubo coincidencia, `search` devuelve `None`
   y `None` no tiene `.group()`. Con 100,000 filas te va a tocar.
4. **`findall` con grupos de captura.** Devuelve los grupos, no la coincidencia
   completa, y si hay dos grupos devuelve tuplas. Si el paréntesis era solo para
   agrupar, usa `(?:...)`.
5. **Comodines en una validación.** `\S+@\S+\.\S+` acepta `$$$@###.!!!` y acepta
   dos arrobas. Y el punto sin escapar, `gmail.com`, acepta `gmailXcom`.


## Qué sigue

**Las semanas 9 y 10: pandas.** Todo lo de esta semana se va a volver una línea,
porque pandas aplica un patrón a una columna completa de un jalón:

| Esta semana, fila por fila | En pandas, columna completa |
|---|---|
| `re.fullmatch(RE_EMAIL, v) is not None` | `df["email"].str.fullmatch(RE_EMAIL)` |
| `re.search(p, v)` | `df["texto"].str.contains(p)` |
| `m.groupdict()` | `df["timestamp"].str.extract(p)` |
| `re.sub(r"\D", "", v)` | `df["telefono"].str.replace(r"\D", "", regex=True)` |

Fíjate en la columna derecha: **el patrón es el mismo**. Lo que cambia es quién
lo aplica. Por eso esta semana va antes y por eso no hay nada que volver a
aprender: lo que escribas en los ejercicios 11 a 16 se reusa tal cual.

**Y de ahí la Práctica 4** (`labs/P4/`), que son las dos cosas juntas: explorar
el dataset con pandas y validar `email` y `telefono` con regex. El diagnóstico de
calidad que armes en el ejercicio 16 es la mitad de su entregable.

Dos cosas que conviene dejar listas desde hoy:

- tu `RE_EMAIL` **probado contra casos malos que escribiste tú**, no solo contra
  los de este notebook
- el criterio de cada columna escrito en una frase, como lo vas a poner en el
  README: qué aceptas, qué rechazas y por qué
